# Event Detection Scoring

**Purpose:** Score EventDetector's pass and interception output against the labelled
ground-truth events, on both the pipeline's predicted possession and a ground-truth
possession oracle, under a matching rule fixed in advance.  
**Inputs:** `data/annotations/event_gt.csv`,
`data/annotations/possession_gt_per_frame.csv`, and the per-clip possession and team
caches in `data/processed/`.  
**Outputs:** `event_scores.csv`, `event_gap_sweep.csv` and
`event_tolerance_sensitivity.csv`, written to `data/outputs/`.  
**Backs:** `results/events/`.

Nothing in this notebook is stochastic: the detector, the oracle construction and the
matcher are all deterministic. The first cell pins the working directory to the repo
root.

In [1]:
import os
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
os.chdir(REPO_ROOT)
print(os.getcwd())

/home/jovyan/nba-video-analytics


## 1. Ground truth and pipeline inputs

Loads the labelled events, builds a per-frame possession oracle from the possession
ground truth (unclear frames read as unheld, which makes the oracle slightly
pessimistic on those 46 frames), and loads each clip's cached possession and team
assignments. A type guard fails fast if the events CSV holds a non-numeric row.

In [2]:
import sys
from dataclasses import dataclass

import pandas as pd
sys.path.insert(0, '.')

from basketball.cache.cache_utils import load_cache
from basketball.events.event_detector import (
    EventDetector, PassEvent, InterceptionEvent, UnclassifiedTransition,
)

CLIPS = ['clip_1', 'clip_2', 'clip_3']

# --- ground truth events ---
gt_events = pd.read_csv('data/annotations/event_gt.csv')
print(f'{len(gt_events)} labelled events')
print(gt_events.groupby(['clip', 'event_type']).size().to_string())

# --- possession ground truth -> oracle possession list, detector's shape ---
poss_gt = (pd.read_csv('data/annotations/possession_gt_per_frame.csv')
             .drop_duplicates(subset=['clip', 'frame_idx'], keep='last'))

# --- pipeline outputs ---
data = {}
for clip in CLIPS:
    possession = load_cache(f'data/processed/{clip}/possession.pkl')
    team = load_cache(f'data/processed/{clip}/team_assignment_fashionclip.pkl')
    n = len(possession)

    # 'unclear' becomes -1, the same value as 'nobody'. This makes the oracle
    # slightly pessimistic on the 46 unclear frames across the three clips: a
    # frame someone may genuinely have held reads as unheld. Stated in the
    # results rather than hidden.
    g = poss_gt[poss_gt['clip'] == clip].set_index('frame_idx')['holder'].to_dict()
    oracle = []
    for f in range(n):
        h = g.get(f)
        oracle.append(-1 if h is None or h in ('nobody', 'unclear') else int(h))

    labelled = sum(1 for f in range(n) if f in g)
    data[clip] = {'possession': possession, 'oracle': oracle, 'team': team, 'n': n}
    print(f'{clip}: {n} frames, {labelled} with possession GT, '
          f'{sum(1 for h in oracle if h != -1)} oracle-held')

# A duplicated header row silently types every column as str, which surfaces
# much later as an arithmetic TypeError in the matcher. Fail here instead.
for col in ('event_id', 'start_frame', 'end_frame'):
    if gt_events[col].dtype == object:
        raise ValueError(
            f'{col} loaded as strings, so the CSV holds a non-numeric row — '
            f'check for a duplicated header. Values: {gt_events[col].tolist()}'
        )

10 labelled events
clip    event_type  
clip_1  block           1
clip_2  pass            4
        shot            1
clip_3  interception    1
        pass            2
        shot            1
clip_1: 117 frames, 117 with possession GT, 89 oracle-held
clip_2: 174 frames, 174 with possession GT, 109 oracle-held
clip_3: 243 frames, 243 with possession GT, 162 oracle-held


## 2. The matcher and its self-checks

Defines the matching rule locked in `docs/prereg/event_detection_spec.md`: a prediction
matches a true event if it falls within tolerance of the labelled interval, has the
representable type, and credits the right team; participant identities are a separate,
stricter criterion. Assignment is greedy one-to-one by temporal proximity. The
known-answer asserts at the end must all pass before any real number is read.

In [3]:
# Ground-truth event type -> the class the detector can emit. Locked in
# docs/prereg/event_detection_spec.md before any results existed.
REPRESENTABLE = {'pass': 'pass', 'interception': 'interception', 'steal': 'interception'}
UNREPRESENTABLE = {'block', 'loose_ball', 'turnover', 'shot', 'inbound', 'other'}


@dataclass(frozen=True)
class Prediction:
    """A detector event flattened to the fields matching compares on."""
    frame_idx: int
    kind: str          # 'pass' or 'interception'
    team: int          # possessing team for a pass, taking team for an interception
    from_id: int
    to_id: int


def flatten(events) -> list[Prediction]:
    """Flatten PassEvent/InterceptionEvent objects into a uniform comparable form."""
    out = []
    for e in events:
        if isinstance(e, PassEvent):
            out.append(Prediction(e.frame_idx, 'pass', e.sender_team,
                                  e.sender_track_id, e.receiver_track_id))
        elif isinstance(e, InterceptionEvent):
            out.append(Prediction(e.frame_idx, 'interception', e.interceptor_team,
                                  e.passer_track_id, e.interceptor_track_id))
    return sorted(out, key=lambda p: p.frame_idx)


def true_team(row) -> int | None:
    """The team a true event is credited to: possessing team for a pass, taking team for an interception."""
    value = row['to_team'] if row['event_type'] in ('interception', 'steal') else row['from_team']
    return None if str(value) == 'unclear' else int(value)


def true_ids(row) -> tuple[int | None, int | None]:
    """The true event's participant track ids, None where unknown."""
    def one(v):
        return None if str(v) == 'unknown' else int(v)
    return one(row['from_track_id']), one(row['to_track_id'])


def qualifies(row, pred: Prediction, tolerance: int, strict_ids: bool) -> bool:
    """Whether a prediction is an admissible match for a true event under the locked rule."""
    if not (row['start_frame'] - tolerance <= pred.frame_idx <= row['end_frame'] + tolerance):
        return False
    if REPRESENTABLE[row['event_type']] != pred.kind:
        return False

    t = true_team(row)
    if t is not None and t != pred.team:
        return False

    if strict_ids:
        f, to = true_ids(row)
        if f is not None and f != pred.from_id:
            return False
        if to is not None and to != pred.to_id:
            return False
    return True


def match(gt_rows: pd.DataFrame, preds: list[Prediction], tolerance: int, strict_ids: bool):
    """Greedy one-to-one assignment by temporal proximity, true events taken in chronological order."""
    unmatched = list(preds)
    pairs, missed = [], []

    for _, row in gt_rows.sort_values('start_frame').iterrows():
        if row['event_type'] in UNREPRESENTABLE:
            continue
        candidates = [p for p in unmatched if qualifies(row, p, tolerance, strict_ids)]
        if not candidates:
            missed.append(row)
            continue
        # Nearest by distance to the interval, not to its midpoint: a prediction
        # inside the interval is distance 0 regardless of where in it it falls.
        best = min(candidates, key=lambda p: max(
            row['start_frame'] - p.frame_idx, p.frame_idx - row['end_frame'], 0))
        unmatched.remove(best)
        pairs.append((row, best))

    return pairs, missed, unmatched


# --- known-answer self-checks: these must all hold before any real number is read ---
def _row(**kw):
    base = {'event_type': 'pass', 'start_frame': 100, 'end_frame': 110,
            'from_team': 1, 'to_team': 1, 'from_track_id': 5, 'to_track_id': 6}
    return pd.Series({**base, **kw})

_gt = pd.DataFrame([_row()])
_p = lambda f, k='pass', t=1, a=5, b=6: Prediction(f, k, t, a, b)

# inside the interval matches
assert len(match(_gt, [_p(105)], 10, False)[0]) == 1
# just outside the interval but inside tolerance matches
assert len(match(_gt, [_p(118)], 10, False)[0]) == 1
# beyond tolerance does not
assert len(match(_gt, [_p(121)], 10, False)[0]) == 0
# wrong type does not
assert len(match(_gt, [_p(105, k='interception')], 10, False)[0]) == 0
# wrong team does not
assert len(match(_gt, [_p(105, t=2)], 10, False)[0]) == 0
# right team, wrong participants: matches on the primary criterion, not the secondary
assert len(match(_gt, [_p(105, a=9, b=8)], 10, False)[0]) == 1
assert len(match(_gt, [_p(105, a=9, b=8)], 10, True)[0]) == 0
# one-to-one: two qualifying predictions, one true event, one left over as a false positive
_pairs, _missed, _fp = match(_gt, [_p(104), _p(106)], 10, False)
assert (len(_pairs), len(_missed), len(_fp)) == (1, 0, 1)
# greedy proximity picks the closer of two, both inside tolerance
_pairs, _, _ = match(_gt, [_p(119), _p(112)], 10, False)
assert _pairs[0][1].frame_idx == 112
# an unrepresentable true event is skipped entirely, never counted as a miss
assert match(pd.DataFrame([_row(event_type='shot')]), [], 10, False)[1] == []

print('matcher self-checks passed')

matcher self-checks passed


## 3. Primary scores

Scores every (clip, arm, tolerance, criterion) combination and writes the grid to
`event_scores.csv`. The locked primary is the team criterion at tolerance 10, with 5
and 20 as sensitivity brackets. Counts are the result; the printed rates only give
them scale.

In [4]:
TOLERANCES = [5, 10, 20]
detector = EventDetector()   # default MAX_TRANSITION_GAP = 30, untuned

raw = {}
for clip in CLIPS:
    d = data[clip]
    for arm, poss in (('predicted', d['possession']), ('oracle', d['oracle'])):
        events = detector._find_transitions(poss, d['team'])
        raw[(clip, arm)] = {
            'preds': flatten(events),
            'unclassified': [e for e in events if isinstance(e, UnclassifiedTransition)],
        }

rows = []
for clip in CLIPS:
    g = gt_events[gt_events['clip'] == clip]
    n_representable = (~g['event_type'].isin(UNREPRESENTABLE)).sum()
    n_unrepresentable = g['event_type'].isin(UNREPRESENTABLE).sum()

    for arm in ('predicted', 'oracle'):
        r = raw[(clip, arm)]
        for tol in TOLERANCES:
            for strict in (False, True):
                pairs, missed, fp = match(g, r['preds'], tol, strict)
                tp, n_fp, n_fn = len(pairs), len(fp), len(missed)
                rows.append({
                    'clip': clip, 'arm': arm, 'tol': tol,
                    'criterion': 'ids' if strict else 'team',
                    'true_events': n_representable, 'unrepresentable': n_unrepresentable,
                    'TP': tp, 'FP': n_fp, 'FN': n_fn,
                    'unclassified': len(r['unclassified']),
                    'precision': round(tp / (tp + n_fp), 3) if tp + n_fp else float('nan'),
                    'recall': round(tp / (tp + n_fn), 3) if tp + n_fn else float('nan'),
                })

scores = pd.DataFrame(rows)
pd.set_option('display.width', 250)

print('=== PRIMARY: team criterion, tolerance 10 ===')
print('Counts are the result; rates are secondary and meaningless without them.\n')
print(scores[(scores.criterion == 'team') & (scores.tol == 10)].to_string(index=False))

print('\n=== Tolerance sensitivity (team criterion) ===')
print(scores[scores.criterion == 'team']
      .pivot_table(index=['clip', 'arm'], columns='tol', values=['TP', 'FP', 'FN'])
      .to_string())

print('\n=== Identity cost: team criterion vs team+participants, tolerance 10 ===')
print(scores[scores.tol == 10]
      .pivot_table(index=['clip', 'arm'], columns='criterion', values='TP')
      .to_string())

scores.to_csv('data/outputs/event_scores.csv', index=False)

=== PRIMARY: team criterion, tolerance 10 ===
Counts are the result; rates are secondary and meaningless without them.

  clip       arm  tol criterion  true_events  unrepresentable  TP  FP  FN  unclassified  precision  recall
clip_1 predicted   10      team            0                1   0   1   0             0      0.000     NaN
clip_1    oracle   10      team            0                1   0   1   0             0      0.000     NaN
clip_2 predicted   10      team            4                1   0   4   4             0      0.000   0.000
clip_2    oracle   10      team            4                1   2   2   2             0      0.500   0.500
clip_3 predicted   10      team            3                1   2   7   1             0      0.222   0.667
clip_3    oracle   10      team            3                1   1   2   2             0      0.333   0.333

=== Tolerance sensitivity (team criterion) ===
                   FN             FP             TP          
tol                5 

## 4. Per-event detail

Every prediction and every miss, per clip and arm, at the primary tolerance: the
listing the aggregate counts compress.

In [5]:
for clip in CLIPS:
    g = gt_events[gt_events['clip'] == clip]
    print(f'\n{"=" * 70}\n{clip}')
    print('GROUND TRUTH')
    print(g[['event_id', 'event_type', 'start_frame', 'end_frame',
             'from_track_id', 'to_track_id', 'from_team', 'to_team']].to_string(index=False))

    for arm in ('predicted', 'oracle'):
        r = raw[(clip, arm)]
        pairs, missed, fp = match(g, r['preds'], 10, False)
        matched = {id(p) for _, p in pairs}
        print(f'\n{arm.upper()} — {len(r["preds"])} classified, '
              f'{len(r["unclassified"])} unclassified')
        for p in r['preds']:
            print(f'  {"MATCH " if id(p) in matched else "FP    "} frame {p.frame_idx:4d}  '
                  f'{p.kind:12s} team {p.team}  {p.from_id} -> {p.to_id}')
        for row in missed:
            print(f'  MISS   event {row["event_id"]} ({row["event_type"]}, '
                  f'{row["start_frame"]}-{row["end_frame"]})')
        if r['unclassified']:
            print('  unclassified at frames: '
                  f'{[e.frame_idx for e in r["unclassified"]]}')


clip_1
GROUND TRUTH
 event_id event_type  start_frame  end_frame  from_track_id to_track_id  from_team to_team
        1      block           85         95              9     unknown          2 unclear

PREDICTED — 1 classified, 0 unclassified
  FP     frame  100  interception team 1  9 -> 8

ORACLE — 1 classified, 0 unclassified
  FP     frame   98  interception team 1  9 -> 8

clip_2
GROUND TRUTH
 event_id event_type  start_frame  end_frame  from_track_id to_track_id  from_team to_team
        1       pass            0         30              9           2          1       1
        2       pass           30         60              2           7          1       1
        3       pass           60         90              7          10          1       1
        4       pass          120        120             10           7          1       1
        5       shot          150        150              7     unknown          1 unclear

PREDICTED — 4 classified, 0 unclassified
  FP     

## 5. Possession-lag context

For each true event, when does the possession ground truth itself first show the
receiver holding, relative to the labelled release? The lag is a property of the
labelling and the confirmation rule, not of EventDetector, and it bounds how early any
possession-based detector could fire.

In [6]:
print('For each true representable event: when does the possession GT itself')
print('first show the receiver holding, versus where the event was labelled?\n')

rows = []
for clip in CLIPS:
    g = gt_events[(gt_events['clip'] == clip) &
                  (~gt_events['event_type'].isin(UNREPRESENTABLE))]
    oracle = data[clip]['oracle']
    preds = {p.frame_idx: p for p in raw[(clip, 'oracle')]['preds']}

    for _, row in g.sort_values('start_frame').iterrows():
        to_id = None if str(row['to_track_id']) == 'unknown' else int(row['to_track_id'])
        # First frame at or after the labelled release where the GT possession
        # list names the receiver. This is the earliest a transition to them
        # could possibly be detected from that list.
        first_held = next(
            (f for f in range(int(row['start_frame']), len(oracle)) if oracle[f] == to_id),
            None,
        )
        nearest_pred = min(preds, key=lambda f: abs(f - int(row['start_frame']))) if preds else None
        rows.append({
            'clip': clip,
            'event_id': row['event_id'],
            'type': row['event_type'],
            'labelled_start': int(row['start_frame']),
            'labelled_end': int(row['end_frame']),
            'receiver': to_id,
            'gt_first_held': first_held,
            'lag_vs_start': None if first_held is None else first_held - int(row['start_frame']),
            'nearest_oracle_pred': nearest_pred,
        })

lag = pd.DataFrame(rows)
print(lag.to_string(index=False))
print('\nlag_vs_start is how far after the labelled release the possession GT')
print('first names the receiver. It is a property of the labelling and the')
print('confirmation rule, not of EventDetector.')

For each true representable event: when does the possession GT itself
first show the receiver holding, versus where the event was labelled?

  clip  event_id         type  labelled_start  labelled_end  receiver  gt_first_held  lag_vs_start  nearest_oracle_pred
clip_2         1         pass               0            30         2             44            44                   44
clip_2         2         pass              30            60         7             62            32                   44
clip_2         3         pass              60            90        10            104            44                   62
clip_2         4         pass             120           120         7            133            13                  133
clip_3         1 interception              30            60         4             88            58                   28
clip_3         2         pass             150           150         2            173            23                  173
clip_3         3   

## 6. Transition-gap sweep

Sweeps EventDetector's max_transition_gap, its one free parameter, on both arms, with
a leave-one-clip-out check on the oracle arm.

In [7]:
GAPS = [10, 20, 30, 45, 60, 90, 120, 240]
gap_rows = []

for gap in GAPS:
    det = EventDetector(max_transition_gap=gap)
    for clip in CLIPS:
        d = data[clip]
        for arm, poss in (('predicted', d['possession']), ('oracle', d['oracle'])):
            preds = flatten(det._find_transitions(poss, d['team']))
            g = gt_events[gt_events['clip'] == clip]
            pairs, missed, fp = match(g, preds, 10, False)
            gap_rows.append({
                'gap': gap, 'clip': clip, 'arm': arm,
                'TP': len(pairs), 'FP': len(fp), 'FN': len(missed),
                'n_preds': len(preds),
            })

gaps = pd.DataFrame(gap_rows)
gaps['f1'] = 2 * gaps.TP / (2 * gaps.TP + gaps.FP + gaps.FN).replace(0, float('nan'))

for arm in ('predicted', 'oracle'):
    print(f'\n=== {arm.upper()} arm, tolerance 10, team criterion ===')
    print(gaps[gaps.arm == arm]
          .pivot_table(index='gap', columns='clip', values=['TP', 'FP', 'FN'])
          .to_string())

print('\n=== Leave-one-clip-out on the gap, oracle arm ===')
print('Selected on the two training clips by pooled F1, reported on the held-out clip.\n')
for held_out in CLIPS:
    train = gaps[(gaps['arm'] == 'oracle') & (gaps['clip'] != held_out)]
    pooled = train.groupby('gap').agg(TP=('TP', 'sum'), FP=('FP', 'sum'), FN=('FN', 'sum'))
    pooled['f1'] = 2 * pooled['TP'] / (2 * pooled['TP'] + pooled['FP'] + pooled['FN'])
    best = pooled['f1'].idxmax()
    te = gaps[(gaps['arm'] == 'oracle') & (gaps['clip'] == held_out) & (gaps['gap'] == best)].iloc[0]
    print(f'held out {held_out}: selected gap={best} '
          f'(train F1 {pooled["f1"].max():.3f}) -> test TP={te.TP} FP={te.FP} FN={te.FN}')

gaps.to_csv('data/outputs/event_gap_sweep.csv', index=False)


=== PREDICTED arm, tolerance 10, team criterion ===
         FN                   FP                   TP              
clip clip_1 clip_2 clip_3 clip_1 clip_2 clip_3 clip_1 clip_2 clip_3
gap                                                                
10      0.0    4.0    2.0    1.0    3.0    5.0    0.0    0.0    1.0
20      0.0    4.0    1.0    1.0    3.0    7.0    0.0    0.0    2.0
30      0.0    4.0    1.0    1.0    4.0    7.0    0.0    0.0    2.0
45      0.0    4.0    1.0    1.0    4.0    7.0    0.0    0.0    2.0
60      0.0    4.0    1.0    1.0    4.0    7.0    0.0    0.0    2.0
90      0.0    4.0    1.0    1.0    4.0    7.0    0.0    0.0    2.0
120     0.0    4.0    1.0    1.0    4.0    7.0    0.0    0.0    2.0
240     0.0    4.0    1.0    1.0    4.0    7.0    0.0    0.0    2.0

=== ORACLE arm, tolerance 10, team criterion ===
         FN                   FP                   TP              
clip clip_1 clip_2 clip_3 clip_1 clip_2 clip_3 clip_1 clip_2 clip_3
gap          

held out clip_2: selected gap=10 (train F1 0.286) -> test TP=1 FP=0 FN=3
held out clip_3: selected gap=20 (train F1 0.444) -> test TP=1 FP=2 FN=2


## 7. Tolerance-rule sensitivity

The gap sweep left `raw` built with the last swept gap, so the first cell rebuilds it
at the default before the tolerance rules are compared. The three rules separate events
not detected at all from events detected but rejected as too late; the fixed tolerance
of 10 stays primary.

In [8]:
detector = EventDetector()   # default gap, matching the primary results
raw = {}
for clip in CLIPS:
    d = data[clip]
    for arm, poss in (('predicted', d['possession']), ('oracle', d['oracle'])):
        events = detector._find_transitions(poss, d['team'])
        raw[(clip, arm)] = {
            'preds': flatten(events),
            'unclassified': [e for e in events if isinstance(e, UnclassifiedTransition)],
        }
print('rebuilt:', sorted(raw.keys()))

rebuilt: [('clip_1', 'oracle'), ('clip_1', 'predicted'), ('clip_2', 'oracle'), ('clip_2', 'predicted'), ('clip_3', 'oracle'), ('clip_3', 'predicted')]


In [9]:
print('Recall under three tolerance rules, oracle arm, team criterion.')
print('The locked +/-10 stays primary; these separate "not detected" from')
print('"detected but rejected as too late".\n')

def match_scaled(gt_rows, preds, rule, strict_ids=False):
    """Greedy one-to-one matching where the tolerance is set per event by rule(row)."""
    unmatched = list(preds)
    pairs, missed = [], []
    for _, row in gt_rows.sort_values('start_frame').iterrows():
        if row['event_type'] in UNREPRESENTABLE:
            continue
        tol = rule(row)
        candidates = [p for p in unmatched if qualifies(row, p, tol, strict_ids)]
        if not candidates:
            missed.append(row)
            continue
        best = min(candidates, key=lambda p: max(
            row['start_frame'] - p.frame_idx, p.frame_idx - row['end_frame'], 0))
        unmatched.remove(best)
        pairs.append((row, best))
    return pairs, missed, unmatched

RULES = {
    'fixed_10 (locked)':  lambda row: 10,
    'fixed_60':           lambda row: 60,
    # A long event is a long uncontrolled period: the interval's own length
    # is the natural scale for how late a confirmation can legitimately be.
    'duration_scaled':    lambda row: max(10, int(row['end_frame'] - row['start_frame'])),
}

rows = []
for name, rule in RULES.items():
    for clip in CLIPS:
        g = gt_events[gt_events['clip'] == clip]
        for arm in ('predicted', 'oracle'):
            pairs, missed, fp = match_scaled(g, raw[(clip, arm)]['preds'], rule)
            rows.append({'rule': name, 'clip': clip, 'arm': arm,
                         'TP': len(pairs), 'FP': len(fp), 'FN': len(missed)})

sens = pd.DataFrame(rows)
for arm in ('predicted', 'oracle'):
    print(f'=== {arm.upper()} ===')
    print(sens[sens['arm'] == arm]
          .pivot_table(index='rule', columns='clip', values=['TP', 'FP', 'FN'])
          .to_string())
    print()

pooled = sens.groupby(['rule', 'arm']).agg(TP=('TP', 'sum'), FP=('FP', 'sum'), FN=('FN', 'sum'))
print('=== Pooled across clips (7 representable events) ===')
print(pooled.to_string())

sens.to_csv('data/outputs/event_tolerance_sensitivity.csv', index=False)

Recall under three tolerance rules, oracle arm, team criterion.
The locked +/-10 stays primary; these separate "not detected" from
"detected but rejected as too late".

=== PREDICTED ===
                      FN                   FP                   TP              
clip              clip_1 clip_2 clip_3 clip_1 clip_2 clip_3 clip_1 clip_2 clip_3
rule                                                                            
duration_scaled      0.0    4.0    1.0    1.0    4.0    7.0    0.0    0.0    2.0
fixed_10 (locked)    0.0    4.0    1.0    1.0    4.0    7.0    0.0    0.0    2.0
fixed_60             0.0    4.0    0.0    1.0    4.0    6.0    0.0    0.0    3.0

=== ORACLE ===
                      FN                   FP                   TP              
clip              clip_1 clip_2 clip_3 clip_1 clip_2 clip_3 clip_1 clip_2 clip_3
rule                                                                            
duration_scaled      0.0    2.0    2.0    1.0    2.0    2.0    0.0  

## 8. Outcome

Three CSVs are written to `data/outputs/`: `event_scores.csv` (the primary grid),
`event_gap_sweep.csv` and `event_tolerance_sensitivity.csv`. The copies shipped with
the repository are in `results/events/`.